In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

print('Current directory:', os.getcwd())
print('Files available:', os.listdir())

Current directory: /Users/isabelladilorenzi/Desktop/University/Semester 1/Hardware and Software for Big Data mod. A/new_sentiment_analysis
Files available: ['sentiment_analysis_testing.ipynb', 'producer.py', '.virtual_documents', 'docker-compose.yml', '.ipynb_checkpoints', 'dataset.csv']


# Sentiment Analysis of Tweets using Apache Spark


The goal of this project is to perform sentiment analysis on textual data using big data technologies. In particular, the task consists of building a multiclass sentiment classification system capable of categorizing tweets into different sentiment classes (positive, negative, uncertainty and litigious).

The dataset used in this project is the Sentiment Dataset with 1 Million Tweets, which contains tweets labeled with sentiment information. Due to the large volume of data and the textual nature of the problem, scalable data processing and machine learning tools are required.

To address this challenge, the project follows the constraints defined in the assignment:

 - Apache Spark with Python (PySpark) is used for data processing and machine learning.

 - Apache Kafka is used as a stream processor to simulate real-time tweet ingestion.

The objective is not only to train an accurate sentiment classification model, but also to evaluate its performance using standard metrics and prepare the system for real-time data processing.

This notebook performs exploratory data analysis and preprocessing on a large-scale tweets dataset using PySpark. The goal is to prepare the data for sentiment analysis by cleaning, filtering, and inspecting sentiment label distributions.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, lower, count, from_json, lit
from pyspark.ml.feature import StringIndexer, Tokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import IndexToString
import json
import pyspark

_kafka_pkg_version = pyspark.__version__
_kafka_packages = (
    f"org.apache.spark:spark-sql-kafka-0-10_2.12:{_kafka_pkg_version},"
    f"org.apache.spark:spark-token-provider-kafka-0-10_2.12:{_kafka_pkg_version}"
)

spark = (SparkSession.builder
    .master("local[*]")
    .appName("H_S_Project")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.jars.packages", _kafka_packages)
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

print(f"SparkSession created successfully! (pyspark {pyspark.__version__}, kafka connector matched)")
spark


26/08/19 16:29:25 WARN Utils: Your hostname, Isabellas-Mac.local resolves to a loopback address: 127.0.0.1; using 192.168.1.45 instead (on interface en0)
26/08/19 16:29:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/isabelladilorenzi/.ivy2/cache
The jars for the packages stored in: /Users/isabelladilorenzi/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.spark#spark-token-provider-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-83d41f1d-5c0b-445c-9aa7-0dc5b8817478;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 194ms :: artifacts dl 7ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#h

SparkSession created successfully! (pyspark 3.5.1, kafka connector matched)


# 1. Dataset and Preprocessing

The dataset consists of tweets along with their associated sentiment labels. Before training the model, several preprocessing steps are applied to clean and prepare the textual data:

 - removal of records with missing values in relevant fields;
 - conversion of text to lowercase;
 - removal of punctuation, numbers, and special characters using regular expressions;
 - tokenization of text into individual words;
 - removal of stopwords to reduce noise in the data.

These steps are implemented using Spark SQL functions and Spark ML feature transformers to ensure scalability and reproducibility.

## 1.1. Loading the Dataset

The dataset is loaded from a CSV file containing tweet text, detected language, and sentiment labels.
Schema inference is enabled to automatically detect column types.


In [6]:
# Load raw data from CSV
df_raw = spark.read.csv(
    'dataset.csv',
    header=True,
    inferSchema=True,
    multiLine=True,
    quote='"',
    escape='"'
)

print('Dataset loaded successfully!')
print(f'Total rows: {df_raw.count()}')
df_raw.printSchema()
df_raw.show(3, truncate=False)

Dataset loaded successfully!


Total rows: 937854
root
 |-- Text: string (nullable = true)
 |-- Language: string (nullable = true)
 |-- Label: string (nullable = true)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------+--------+---------+
|Text                                                                                                                                                       |Language|Label    |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------+--------+---------+
|@Charlie_Corley @Kristine1G @amyklobuchar @StyleWriterNYC testimony is NOT evidence in a court of law, state or federal. Must stand up to cross examination|en      |litigious|
|#BadBunny: Como dos gotas de agua: Joven se disfraza de Bad Bunny y causa tumulto en alfombra roja. https://t.co/3524SEangh                              

## 1.2. Inspecting the Raw Data

We inspect the schema and a sample of rows to verify that the data was loaded correctly.


In [8]:
# This confirms columns (Text, Language, Label)
df_raw.printSchema()
df_raw.show(5, truncate=False)


root
 |-- Text: string (nullable = true)
 |-- Language: string (nullable = true)
 |-- Label: string (nullable = true)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+---------+
|Text                                                                                                                                                                                                                                                                                                           |Language|Label    |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 1.3. Data Cleaning

The dataset is cleaned by:
- Removing rows with missing sentiment labels
- Removing rows with missing language information
- Preserving the original tweet text for further NLP processing


In [10]:
# Clean data: remove nulls and clean text
df_clean = (
    df_raw
    .dropna(subset=['Label', 'Language'])
    .withColumn('clean_text', 
            lower(
             regexp_replace(               # remove non letters
                regexp_replace(            # remove www links
                    regexp_replace(        # remove http/https links
                        col("Text"),
                        r"http\S+",
                        ""
                    ),
                    r"www\.\S+",
                    ""
                ),
                r"[^a-zA-Z\s]",
                ""
            )
        )
    )
)

print(f'Cleaned rows: {df_clean.count()}')
df_clean.select('Text', 'clean_text', 'Label').show(5, truncate=60)

Cleaned rows: 937831
+------------------------------------------------------------+------------------------------------------------------------+---------+
|                                                        Text|                                                  clean_text|    Label|
+------------------------------------------------------------+------------------------------------------------------------+---------+
|@Charlie_Corley @Kristine1G @amyklobuchar @StyleWriterNYC...|charliecorley kristineg amyklobuchar stylewriternyc testi...|litigious|
|#BadBunny: Como dos gotas de agua: Joven se disfraza de B...|badbunny como dos gotas de agua joven se disfraza de bad ...| negative|
|https://t.co/YJNiO0p1JV Flagstar Bank discloses a data br...| flagstar bank discloses a data breach that impacted \nmi...|litigious|
|Rwanda is set to host the headquarters of United Nations ...|rwanda is set to host the headquarters of united nations ...| positive|
|OOPS. I typed her name incorrectly (toda

## 1.4. Sentiment Label Distribution

We analyze the distribution of sentiment labels to understand class balance in the dataset.
This step ensures that the `Label` column contains only sentiment values.


In [12]:
# Sentiment distribution
print('=== Sentiment Distribution ===')
df_clean.groupBy('Label').count().orderBy('count', ascending=False).show()

=== Sentiment Distribution ===


+-----------+------+
|      Label| count|
+-----------+------+
|   positive|264539|
|   negative|262208|
|uncertainty|206940|
|  litigious|204144|
+-----------+------+



## 1.5. Language Distribution

This analysis shows the most common languages present in the dataset.


In [14]:
# Language distribution
print('\n=== Language Distribution ===')
df_clean.groupBy('Language').count().orderBy('count', ascending=False).show(10)


=== Language Distribution ===


+--------+------+
|Language| count|
+--------+------+
|      en|871310|
|      fr| 13091|
|      es| 11333|
|      pt| 10336|
|      ja|  8414|
|      in|  3000|
|      tl|  2816|
|     und|  2702|
|      de|  2055|
|      tr|  1371|
+--------+------+
only showing top 10 rows



## 1.6. English Subset

For downstream NLP tasks, we restrict the dataset to English tweets only.


In [16]:
# Keep only English tweets for NLP consistency
df_en = (
    df_clean
    .filter(col('Language') == 'en')
    .select(col('clean_text'), col('Label'))
)

print(f'English tweets: {df_en.count()}')
df_en.show(3, truncate=100)

English tweets: 871310
+----------------------------------------------------------------------------------------------------+---------+
|                                                                                          clean_text|    Label|
+----------------------------------------------------------------------------------------------------+---------+
|charliecorley kristineg amyklobuchar stylewriternyc testimony is not evidence in a court of law s...|litigious|
|             flagstar bank discloses a data breach that impacted \nmillion individuals cybersecurity|litigious|
|rwanda is set to host the headquarters of united nations development programmes undp new innovati...| positive|
+----------------------------------------------------------------------------------------------------+---------+
only showing top 3 rows



## 1.7. Dataset for Machine Learning

We keep only the text and sentiment label needed for classification.


In [18]:
from pyspark.sql.functions import col

df_ml = df_en.select(
    col("clean_text"),
    col("Label").alias("label")
)

df_ml.printSchema()

root
 |-- clean_text: string (nullable = true)
 |-- label: string (nullable = true)



## 1.8. Label Encoding

Spark ML requires the target variable to be numeric.
We convert sentiment labels into numerical indices.


In [20]:
# Encode sentiment labels to numeric values
label_indexer = StringIndexer(
    inputCol='Label',
    outputCol='label_index',
    handleInvalid="skip"
)

label_indexer_model = label_indexer.fit(df_en)
df_indexed = label_indexer_model.transform(df_en)

print('Label Encoding:')
print('Positive -> 0.0')
print('Negative -> 1.0')
print('Uncertainty -> 2.0')
print('Litigious -> 3.0')
print()
df_indexed.select('Label', 'label_index').distinct().show()

Label Encoding:
Positive -> 0.0
Negative -> 1.0
Uncertainty -> 2.0
Litigious -> 3.0



+-----------+-----------+
|      Label|label_index|
+-----------+-----------+
|   negative|        1.0|
|uncertainty|        2.0|
|   positive|        0.0|
|  litigious|        3.0|
+-----------+-----------+



## 1.9. Tokenization & Stopword Removal

We split each tweet into individual words.


In [22]:
# Tokenization - split text into words
tokenizer = Tokenizer(
    inputCol='clean_text',
    outputCol='tokens'
)

# Stopword Removal - remove common English words
remover = StopWordsRemover(
    inputCol='tokens',
    outputCol='filtered_tokens'
)

print('Tokenizer and StopWordsRemover initialized')

Tokenizer and StopWordsRemover initialized


# 2 Feature Extraction

To convert textual data into numerical representations suitable for machine learning, a TF-IDF (Term Frequency–Inverse Document Frequency) approach is adopted:

- CountVectorizer is used to generate term-frequency vectors from the cleaned tokens;
- IDF is applied to weight terms based on their importance across the dataset.

This approach helps emphasize informative words while reducing the impact of very common terms.

## 2.1. Train–Test Split

The dataset is split into:
- 80% training data to train the model
- 20% test data to evaluate how well the model generalizes to unseen data


In [24]:
# Split data into training (80%) and testing (20%)
train_df, test_df = df_indexed.randomSplit([0.8, 0.2], seed=42)

print(f'Training samples: {train_df.count()}')
print(f'Testing samples: {test_df.count()}')

Training samples: 697501


Testing samples: 173809


## 2.2. Text Vectorization (TF-IDF)

We convert text into numerical feature vectors using TF-IDF.

TF-IDF stands for:

- TF → Term Frequency

- IDF → Inverse Document Frequency

The idea:

- Words that appear in many documents (like “the”, “and”, “is”) get down‑weighted

- Words that appear in few documents (like “asphyxiation”, “blockchain”, “neapolitan”) get up‑weighted

So TF‑IDF highlights important, discriminative words.

In [26]:
# CountVectorizer - convert tokens to term frequency vectors
cv = CountVectorizer(
    inputCol='filtered_tokens',
    outputCol='raw_features',
    minDF=2
)

# IDF - apply inverse document frequency weighting
idf = IDF(
    inputCol='raw_features',
    outputCol='features'
)

print('TF-IDF components initialized')

TF-IDF components initialized


# 3. Machine Learning Model

A Logistic Regression classifier is used to perform multiclass sentiment classification. Logistic Regression is a widely adopted and interpretable model that integrates well with Spark ML pipelines.

All preprocessing, feature extraction, and classification steps are combined into a single Spark ML Pipeline, ensuring a clean and modular workflow. The dataset is split into training and test sets to evaluate the generalization performance of the model.

So, now that the text data has been transformed into numerical TF-IDF feature vectors,  
we can train a machine learning model to predict sentiment.

The following steps will be performed:
- Train a classification model
- Generate predictions
- Evaluate model performance

## 3.1. Logistic Regression Model

Logistic Regression is a commonly used classification algorithm for text data.
It learns a linear decision boundary based on the TF-IDF feature vectors.


In [29]:
# Initialize Logistic Regression classifier
lr = LogisticRegression(
    featuresCol='features',
    labelCol='label_index',
    family='multinomial',
    maxIter=20,
    regParam=0.01
)

## 3.2 Pipeline Integration: All stages assembled in order

In [31]:
# Create pipeline with all stages
pipeline = Pipeline(stages=[
    tokenizer,          # Stage 0: Tokenize text
    remover,            # Stage 1: Remove stopwords
    cv,                 # Stage 2: Count vectorization
    idf,                # Stage 3: TF-IDF weighting
    lr                  # Stage 4: Logistic Regression
])

print('Pipeline created. Starting training...')
pipeline_model = pipeline.fit(train_df)
print('Training complete!')

Pipeline created. Starting training...


Training complete!


#### Generating predictions on the test set

In [33]:
# Generate predictions on test set
predictions = pipeline_model.transform(test_df)

print('Predictions generated!')
predictions.select('clean_text', 'label_index', 'prediction', 'probability').show(10, truncate=90)

Predictions generated!


+------------------------------------------------------------------------------------------+-----------+----------+------------------------------------------------------------------------------------+
|                                                                                clean_text|label_index|prediction|                                                                         probability|
+------------------------------------------------------------------------------------------+-----------+----------+------------------------------------------------------------------------------------+
|                                                           \n\n\n\n\n\n\n\n\n\ngood night |        0.0|       0.0|  [0.8697829989845243,0.05163630686841277,0.048760483786647144,0.029820210360415853]|
|\n\n\n\n years later present day\n\ntsuki tsuki\n\nbakugou was jostled awake by his bes...|        2.0|       2.0|     [0.1780054622413164,0.13456165893962574,0.4782946531340946,0.209138225684963

#### Extract Trained Transformers for streaming use

In [35]:
# Extract trained transformers for streaming use
tokenizer_trained = pipeline_model.stages[0]
remover_trained = pipeline_model.stages[1]
cv_trained = pipeline_model.stages[2]
idf_trained = pipeline_model.stages[3]
lr_trained = pipeline_model.stages[4]

# Converts the model's numeric prediction (0.0, 1.0, 2.0, 3.0) back into the
# original string labels ("positive", "negative", "uncertainty", "litigious").
# Needed by predict_batch() in the streaming section below.
from pyspark.ml.feature import IndexToString
label_decoder = IndexToString(
    inputCol="prediction",
    outputCol="sentiment",
    labels=label_indexer_model.labels
)

print('All model stages extracted successfully!')


All model stages extracted successfully!


## 3.3. Model Evaluation

Model performance is evaluated using: **accuracy**, which measures the proportion
of correctly classified instances in the test set; **weighted precision**, that measures how reliable the predicted sentiment labels are; **weighted recall**, which measures how well the model identifies all tweets belonging to each sentiment class.

**Confusion matrix** was performed by grouping the batch predictions by true labels and predicted labels. The resulting counts show that most instances lie on the diagonal, indicating a high number of correct classifications across all sentiment classes.

In [37]:
# Evaluate model performance
evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol='label_index',
    predictionCol='prediction',
    metricName='accuracy'
)

accuracy = evaluator_accuracy.evaluate(predictions)

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol='label_index',
    predictionCol='prediction',
    metricName='weightedPrecision'
)

precision = evaluator_precision.evaluate(predictions)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol='label_index',
    predictionCol='prediction',
    metricName='weightedRecall'
)

recall = evaluator_recall.evaluate(predictions)

print('=' * 50)
print('MODEL EVALUATION RESULTS')
print('=' * 50)
print(f'Accuracy:          {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'Weighted Precision: {precision:.4f} ({precision*100:.2f}%)')
print(f'Weighted Recall:   {recall:.4f} ({recall*100:.2f}%)')
print('=' * 50)

# Confusion matrix
conf_matrix = (
    predictions.groupBy("label_index", "prediction")
    .count()
    .groupBy("label_index")
    .pivot("prediction")
    .sum("count")
    .orderBy("label_index")
    .fillna(0)
)

matrix_rows = conf_matrix.collect()
pred_labels = conf_matrix.columns[1:]

print('=' * 50)
print('CONFUSION MATRIX')
print('=' * 50)
print(f"{'Label':<10}" + "".join([f"{str(p):>10}" for p in pred_labels]))
print('-' * 50)

for row in matrix_rows:
    label = row['label_index']
    counts = [row[p] for p in pred_labels]
    print(f"{str(label):<10}" + "".join([f"{c:>10}" for c in counts]))

print('=' * 50)


MODEL EVALUATION RESULTS
Accuracy:          0.9557 (95.57%)
Weighted Precision: 0.9558 (95.58%)
Weighted Recall:   0.9557 (95.57%)


CONFUSION MATRIX
Label            0.0       1.0       2.0       3.0
--------------------------------------------------
0.0            47580       964       781       351
1.0              994     46337       859       543
2.0              682       684     37949       217
3.0              496       704       418     34250


### Results

The Logistic Regression classifier achieved a test accuracy, weighted precision and weighted recall of **95.9%**.
This indicates that the TF-IDF feature representation effectively captures
sentiment-related information in the text data.

Misclassifications are relatively limited and mainly occur between neighboring classes, suggesting that the model captures meaningful sentiment patterns. This analysis complements standard evaluation metrics by providing insight into class-level prediction behavior.

# 4. Kafka Streaming

In [40]:
# Read real tweets from Kafka in real time.
# Start Kafka first (see docker-compose.yml) and run producer.py to publish
# tweets into the "tweets-input" topic before running this cell.
KAFKA_BOOTSTRAP_SERVERS = "localhost:9092"
KAFKA_TOPIC = "tweets-input"

stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
    .selectExpr("CAST(value AS STRING) AS Text")
    .withColumn("Language", lit("en"))
)

stream_df.printSchema()
print(f"Reading from Kafka topic '{KAFKA_TOPIC}' on {KAFKA_BOOTSTRAP_SERVERS}")


root
 |-- Text: string (nullable = true)
 |-- Language: string (nullable = false)

Reading from Kafka topic 'tweets-input' on localhost:9092


Text cleaning, tokenization, TF-IDF and prediction for the stream are all
handled inside `predict_batch` below by re-using the already-fitted
`pipeline_model`, so we don't need to rebuild the pipeline stage-by-stage
here (that was dead code in the original version, since `predict_batch`
re-implemented the same steps separately).

In [43]:
def predict_batch(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return

    batch_clean = (
        batch_df
        .filter(col("Language") == "en")
        .withColumn(
            "clean_text",
            lower(
                regexp_replace(
                    regexp_replace(
                        regexp_replace(col("Text"), r"http\S+", ""),
                        r"www\.\S+", ""
                    ),
                    r"[^a-zA-Z\s]", ""
                )
            )
        )
    )

    # Reuse the SAME fitted pipeline used for batch evaluation
    # (tokenizer -> stopword removal -> CountVectorizer -> IDF -> LogisticRegression)
    batch_predictions = pipeline_model.transform(batch_clean)
    batch_final = label_decoder.transform(batch_predictions)

    print(f"--- micro-batch {batch_id} ({batch_final.count()} rows) ---")
    batch_final.select("Text", "sentiment", "probability").show(truncate=100)


In [44]:
# Write streaming predictions to console and start query.
query = (
    stream_df
    .writeStream
    .foreachBatch(predict_batch)
    .option("checkpointLocation", "/tmp/spark_checkpoint_kafka")
    .trigger(processingTime="5 seconds")
    .start()
)

query.awaitTermination(timeout=120)  # runs for 2 minutes, then stops cleanly
query.stop()
print("Streaming query stopped.")


--- micro-batch 25 (2 rows) ---
+----------------------------------------------------------------------------------------------------+---------+------------------------------------------------------------------------------------+
|                                                                                                Text|sentiment|                                                                         probability|
+----------------------------------------------------------------------------------------------------+---------+------------------------------------------------------------------------------------+
|Have you ever found a baby wild animal and not known what to do it? That’s why The Arboretum is e...| positive|[0.9693040312837468,0.018356280355113795,0.009941582085431328,0.0023981062757081287]|
|.@Lebeaucarnews It's great to see the #GM brands improve in the @JDPowerAutos initial quality sur...| positive|  [0.988904359882412,0.005276555690424896,0.004042583285639073,0

+----------------------------------------------------------------------------------------------------+-----------+-----------------------------------------------------------------------------------+
|                                                                                                Text|  sentiment|                                                                        probability|
+----------------------------------------------------------------------------------------------------+-----------+-----------------------------------------------------------------------------------+
|@TWLadyGrey @jm_pango @SpecialPuppy1 @SenSanders I know people from Yerevan who did not have the ...|   negative|  [0.22274787216090733,0.7481567433830089,0.004601653387922379,0.02449373106816143]|
|                                   @YAW_OB There’s nothing wrong with having different opinions thou|   negative| [0.017018235862839258,0.9503744777915857,0.01608951168092864,0.016517774664646476]|
|@ANT

+----------------------------------------------------------------------------------------------------+---------+-------------------------------------------------------------------------------------+
|                                                                                                Text|sentiment|                                                                          probability|
+----------------------------------------------------------------------------------------------------+---------+-------------------------------------------------------------------------------------+
|                                                                I’m down bad https://t.co/yi5ToIrbgY| negative|   [0.045103990415408385,0.8948630029823582,0.03593854166004053,0.024094464942192813]|
|@ChuckTingle I get ya, bud. I believe in my PowerPoint decks the same way. \n\n‘cept my decks ain...| negative| [0.009690378602588774,0.9858152313299126,8.610207105479367E-4,0.0036333693569508218]|
|@CH0

+----------------------------------------------------------------------------------------------------+---------+----------------------------------------------------------------------------------+
|                                                                                                Text|sentiment|                                                                       probability|
+----------------------------------------------------------------------------------------------------+---------+----------------------------------------------------------------------------------+
|                   Good start - time to keep the pressure on! 🔥\n\n#U20MYNT https://t.co/uKZ2ykDl2O| positive|   [0.8448349482594402,0.0538255612919274,0.07064848200398173,0.03069100844465057]|
|Shut the fuck up for once dude is making opting into a contract seem like something revolutionary...|litigious|[0.027226399010135984,0.01850727737459094,0.032334869955462385,0.9219314536598108]|
+--------------------

--- micro-batch 39 (3 rows) ---
+----------------------------------------------------------------------------------------------------+---------+-----------------------------------------------------------------------------------+
|                                                                                                Text|sentiment|                                                                        probability|
+----------------------------------------------------------------------------------------------------+---------+-----------------------------------------------------------------------------------+
|@CheriJacobus @lexikav @CallieKhouri Maybe because she was waiting for protection. She is young a...|litigious|[6.547819428465651E-4,0.006488225089779568,0.015806676770246973,0.9770503161971269]|
|@baileymaet14 @rileyhicks14 what you’re saying makes sense and i agree but how do you know the va...| negative| [0.004689704564980107,0.9418053918610544,0.007121184303329437,0.046

+----------------------------------------------------------------------------------------------------+---------+------------------------------------------------------------------------------------+
|                                                                                                Text|sentiment|                                                                         probability|
+----------------------------------------------------------------------------------------------------+---------+------------------------------------------------------------------------------------+
|@mikepompeo The adoption process does jack shit. It's just as flawed as our corrections system an...| negative|    [0.011599759665103656,0.7907054593822485,0.005146490812566877,0.192548290140081]|
|RTHC completed a handrail installation project for a disabled homeowner who had a leg amputated a...| positive|[0.9852065041265701,2.1177480845196073E-4,0.006389808190532139,0.008191912874445735]|
+---------

+----------------------------------------------------------------------------------------------------+---------+-------------------------------------------------------------------------------------+
|                                                                                                Text|sentiment|                                                                          probability|
+----------------------------------------------------------------------------------------------------+---------+-------------------------------------------------------------------------------------+
|@AndyisLive With many out of contract players we have room in the wages bill, maybe no transfer f...|litigious|    [0.00448790169398279,0.007690111324249084,0.13197134642803035,0.8558506405537377]|
|Lord Phillips, who was master of the rolls &amp; led the first @londonlegal #LegalWalk in 2005, w...|litigious|[5.580812525717834E-5,1.0942582552807932E-5,3.9761046662435344E-5,0.9998934882455276]|
+----

--- micro-batch 48 (2 rows) ---


+----------------------------------------------------------------------------------------------------+---------+--------------------------------------------------------------------------------+
|                                                                                                Text|sentiment|                                                                     probability|
+----------------------------------------------------------------------------------------------------+---------+--------------------------------------------------------------------------------+
|@RolexOyugi @AJEnglish He just isn’t, they lie about everything but people believe them when they...| negative|[0.01786334800241785,0.8683019595841667,0.02352283766872911,0.09031185474468631]|
|                                           今年はムカデに1匹も会わずに梅雨が終わっちまったな！(慢心)| positive|     [0.3538986969486964,0.2960808103605,0.2089230495169465,0.14109744317385717]|
+--------------------------------------------------------

# Project Summary

## Project Overview
This notebook implements a multiclass sentiment classification system using Apache Spark and Logistic Regression.

### Dataset
- **Source**: Kaggle - Sentiment Dataset with 1 Million Tweets
- **Classes**: 4 (Positive, Negative, Uncertainty, Litigious)
- **Language**: English-only (filtered)

### Model Performance
- **Accuracy**: 95.9%
- **Weighted Precision**: 95.9%
- **Weighted Recall**: 95.9%

### Technology Stack
- Apache Spark 3.5.1
- PySpark ML
- Logistic Regression
- TF-IDF Feature Extraction
- Apache Kafka (optional streaming)

### Pipeline Stages
1. **Tokenization**: Split text into words
2. **Stopword Removal**: Remove common English words
3. **Count Vectorization**: Convert tokens to term frequency vectors
4. **IDF**: Apply inverse document frequency weighting
5. **Label Encoding**: Convert sentiment labels to numeric values
6. **Logistic Regression**: Train multiclass classifier